#### DAY 3

In [1]:
import pandas as pd

In [5]:
df = pd.read_csv('../data/hotel_bookings.csv')

df.shape #cheking the rows and cols

(119390, 32)

#### Identifying the missing values

In [6]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

#### Calculating the missing value percentage

In [10]:
missing_percentage = (df.isnull().sum() / len(df)) * 100 #to know the % of missing value

missing_percentage[missing_percentage > 0] # this selects only the columns that has missing values

children     0.003350
country      0.408744
agent       13.686238
company     94.306893
dtype: float64

#### Starting to handle the missing values in CHILDREN column

In [12]:
df[df['children'].isnull()] #identifies the rows which has no of children missing witht the full complete record

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40600,City Hotel,1,2,2015,August,32,3,1,0,2,...,No Deposit,NaN,NaN,0,Transient-Party,12.0,0,1,Canceled,2015-08-01
40667,City Hotel,1,1,2015,August,32,5,0,2,2,...,No Deposit,14.0,NaN,0,Transient-Party,12.0,0,1,Canceled,2015-08-04
40679,City Hotel,1,1,2015,August,32,5,0,2,3,...,No Deposit,NaN,NaN,0,Transient-Party,18.0,0,2,Canceled,2015-08-04
41160,City Hotel,1,8,2015,August,33,13,2,5,2,...,No Deposit,9.0,NaN,0,Transient-Party,76.5,0,1,Canceled,2015-08-09


In [15]:
df[df['children'].isnull()][['adults', 'children', 'babies']]

#but we cant come to the conclusion that having no babies doesnt means they adults have children

,adults,children,babies
40600,2,NaN,0
40667,2,NaN,0
40679,3,NaN,0
41160,2,NaN,0


#### checking other distribution of children

In [17]:
df['children'].value_counts(dropna=False).sort_index()

#value_counts() counts how often each number of children appears.
#dropna=False includes the four missing values in the output.
#sort_index() sorts the results by the number of children.

children
0.0     110796
1.0       4861
2.0       3652
3.0         76
10.0         1
NaN          4
Name: count, dtype: int64

Above study, we discovered around 92.8% bookings have 0 children

only 4 bookings have mssing values
and one booking has 10 children which is unusual but we can check it later

#### CONCLUSION

We came to the conclusion that, we will be handling the missing values in children using 'MODE IMPUTATION'

MODE is the most frequently occuring value in a colmn, here its 0

its used here bcs missing proportion is extremely small here.

In [18]:
df['children'].mode()

0    0.0
Name: children, dtype: float64

#### Starting to handle the missing values in COUNTRY column

In [21]:
df['country'].value_counts(dropna=False).head(10)

country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
ITA     3766
IRL     3375
BEL     2342
BRA     2224
NLD     2104
Name: count, dtype: int64

#### checking missing country records

In [23]:
df[df['country'].isnull()][
    ['hotel','country','is_canceled']
].head(10)

#We want to confirm that these records still contain useful information
#including hotel type and cancellation status.

#If they do, retaining them with an Unknown country category is a reasonable approach.

,hotel,country,is_canceled
30,Resort Hotel,NaN,0
4127,Resort Hotel,NaN,1
7092,Resort Hotel,NaN,1
7860,Resort Hotel,NaN,1
8779,Resort Hotel,NaN,1
9376,Resort Hotel,NaN,1
9610,Resort Hotel,NaN,1
9612,Resort Hotel,NaN,1
11262,Resort Hotel,NaN,1
11263,Resort Hotel,NaN,1


#### CONCLUSION

here, for the country column we came to the conclusion that we will handle the missing value using 'Unknown'

This is preferable to filling with PRT (Portugal), because we don't know the actual countries.

We will apply this strategy when building our cleaning and preprocessing pipeline.

#### Starting to handle the missing values in AGENT column

The agent column represents the ID of the travel agency associated with a booking.

A missing value may indicate that no travel agency was involved, but we should not assume this without checking.

In [25]:

df['agent'].value_counts(dropna=False).head(10)

#here, the agent id is just identities not quantities eg id9

agent
9.0      31961
NaN      16340
240.0    13922
1.0       7191
14.0      3640
7.0       3539
6.0       3290
250.0     2870
241.0     1721
28.0      1666
Name: count, dtype: int64

Here, it means that agent 9 is most frequent because it has 31,961 bookings total
also 16,340 bookings without an agent ID

Another concept we found was, agent column is actually categorical, even tho it contains number because agent 240 isnt greater or important than agent 9,these are identifctn codes not numericals

In [28]:
df['agent'].nunique()

#checking how many unique agents exists 
#Nunique() is used to count number of distincts agents ID

#If there are many different agent IDs, using one-hot encoding could create a large number of new columns.

333

#### CONCLUSION

I recommend creating a simpler feature:

has_agent
- 1 = An agent ID is recorded.
- 0 = Agent ID is missing.
This preserves potentially useful information without introducing hundreds of columns. Importantly, 0 means missing agent ID, not necessarily no travel agent.

Suppose an agent ID is missing.

If we replace it with 9 (the most common agent), we're assuming that booking was handled by Agent 9, even though we don't know that.

Similarly, calculating the mean of agent IDs wouldn't make sense because agent IDs are identifiers, not measurements.

#### Starting to handle the missing values in AGENT column


In [29]:
df['company'].value_counts(dropna=False).head(10)
#examining 

company
NaN      112593
40.0        927
223.0       784
67.0        267
45.0        250
153.0       215
174.0       149
219.0       141
281.0       138
154.0       133
Name: count, dtype: int64

what we understood here is, 112593 bookings have zero compny ID, same like agents

#### CONCLUSION

Instead of filling the missing company IDs, with the mean and mode, creating a new feature called has_company

In [31]:

df['company'].nunique()


352

#### INVESTIGATING DUPLICATE RECORDS

During Day 1, i discovered that our dataset contains 31,994 duplicate rows
thats 26.68% of the dataset

but not every duplicate looking row can be duplicate booking 

In [38]:
df[df.duplicated(keep=False)].sort_values(
    by=['hotel', 'lead_time']
).head(10)

#code explanation
# df.duplicated(keep=False) -> identifying all the rows that belongs to duplicate grps
#sorting the values by hotel andd leadtime

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40643,City Hotel,1,0,2015,August,32,4,0,1,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Canceled,2015-08-04
40644,City Hotel,1,0,2015,August,32,4,0,1,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Canceled,2015-08-04
40772,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09
40802,City Hotel,0,0,2015,August,32,7,0,2,2,...,No Deposit,14.0,NaN,0,Transient,75.0,0,1,Check-Out,2015-08-09
40821,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09
40838,City Hotel,0,0,2015,August,32,8,0,1,2,...,No Deposit,9.0,NaN,0,Transient,89.0,0,1,Check-Out,2015-08-09
41026,City Hotel,0,0,2015,August,33,11,0,1,2,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-08-12
41067,City Hotel,0,0,2015,August,33,11,0,1,1,...,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41071,City Hotel,0,0,2015,August,33,11,0,1,2,...,No Deposit,NaN,NaN,0,Transient,80.0,0,0,Check-Out,2015-08-12
41073,City Hotel,0,0,2015,August,33,11,0,1,1,...,No Deposit,NaN,38.0,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12


#### Check duplicate distribution by cancellation status

In [36]:
df[df.duplicated()].groupby('is_canceled').size()


#used to count the duplicte bookings seprtly for canceled and non canceled 
#df.duplicated() -> identifies repeated rows
#df[df.duplicated()] -> selects only those duplicte rows
#groupby('is_canceled') -> grps only rhe canctn sts
#.size() -> counts the rows in each grp

is_canceled
0    11795
1    20199
dtype: int64

#### OBSERVATION

Booking status	Total bookings	Duplicate rows	Duplicate percentage
Not canceled	75,166	11,795	15.69%
Canceled	44,224	20,199	45.67%

However, we cannot conclude that these records are errors or that canceled bookings are inherently more likely to be duplicated.

I'll retain the duplicate-looking records during data-quality analysis, because we cannot reliably identify separate bookings without a unique booking ID.

#### Investigating unusual values 

During Day 1, we noticed that some bookings had zero guests.

A hotel booking normally needs at least one guest, so we need to investigate these records.


In [41]:
df[
    (df['adults'] == 0) &
    (df['children'] == 0) &
    (df['babies'] == 0)
].shape[0]


# this code means that checking for zero adults babies and children ie 0 guests

180

#### Check cancellation status of zero-guest bookings

In [42]:

df[
    (df['adults'] == 0) &
    (df['children'] == 0) &
    (df['babies'] == 0)
]['is_canceled'].value_counts()


is_canceled
0    155
1     25
Name: count, dtype: int64

#### cleaning decision:

We'll exclude these 180 records from our modeling dataset, assuming they represent invalid guest counts. This removes only approximately 0.15% of the original dataset.


In [43]:

df_clean = df.copy() #creates the copy of org dataset
 
df_clean['total_guests'] = df_clean[['adults', 'children', 'babies']].sum(axis=1, min_count=3) #to calculate the total guests,also if any guest is missing totalguest stays missing rather than assuming its zero

df_clean = df_clean[df_clean['total_guests'] != 0] # keepng the booking where total-guests !=0

df_clean.shape #displays remaining rows and cols




(119210, 33)

After cleaning we got 11920 rows and 33 cols and 180 zero guests are removes

#### Investigating unusual room prices

During Day 1, we noticed some unusual values in the adr column.

ADR stands for Average Daily Rate. It represents the average room revenue per occupied room-night.

For example, if the ADR is 150, the recorded average daily rate is 150 currency units.

We previously noticed:
- Minimum ADR: -6.38
- Maximum ADR: 5400
A negative room rate is unusual, so let's investigate it before deciding what to do.

In [46]:

df_clean[df_clean['adr'] < 0][['hotel', 'adr', 'is_canceled']]

#df_clean['adr'] < 0 checks for negative ADR values.
#df_clean[...] filters the rows containing negative ADR.
#- [['hotel', 'adr', 'is_canceled']] displays only those three columns.

,hotel,adr,is_canceled
14969,Resort Hotel,-6.38,0


We want to see how many bookings have negative room rates and whether they were canceled.

For this project, I am removing this one record from my modeling dataset because its negative ADR is not representative of a normal room rate. 

In [48]:
df_clean = df_clean[df_clean['adr'] >= 0].copy()

df_clean.shape

#removed the negative adr value

(119209, 33)

TOTAL ROWS REMOVED : 181

#### Investigating extremely high ADR values

In [49]:
df_clean[df_clean['adr'] > 1000][['hotel', 'adr', 'is_canceled']]


,hotel,adr,is_canceled
48515,City Hotel,5400.0,1


I discovered that only one data is above 1000 ie 5400 adr

this looks like an outlier

In [50]:
df_clean.loc[48515]


hotel                                City Hotel
is_canceled                                   1
lead_time                                    35
arrival_date_year                          2016
arrival_date_month                        March
arrival_date_week_number                     13
arrival_date_day_of_month                    25
stays_in_weekend_nights                       0
stays_in_week_nights                          1
adults                                        2
children                                    0.0
babies                                        0
meal                                         BB
country                                     PRT
market_segment                    Offline TA/TO
distribution_channel                      TA/TO
is_repeated_guest                             0
previous_cancellations                        0
previous_bookings_not_canceled                0
reserved_room_type                            A
assigned_room_type                      

The above bookings looks unusual

- Two adults booked a room.
- The stay was for one night.
- The booking was made 35 days before arrival.
- The booking was eventually canceled.
However, an ADR of 5,400 is extremely high compared with the rest of our dataset.

I found no other booking with an ADR above 1,000.

I am removing this outlier because it has extreme adr which can affect the model, but not all outlers should be removed but this we investagated and justified

#### Remove the extreme adr record

In [51]:
df_clean = df_clean[df_clean['adr'] <= 1000].copy()

df_clean.shape

(119208, 33)

we have removed 182 record from the table 

#### Investigating unusually high guest counts

During the day, adults cols had maximum value of 55 which we found as unusual

In [53]:
df_clean[df_clean['adults'] > 10][
    ['hotel', 'adults', 'children', 'babies', 'total_guests', 'is_canceled']
]

#why we took 10, its and explortory threshold to help us inspct unusual booing

,hotel,adults,children,babies,total_guests,is_canceled
1539,Resort Hotel,40,0.0,0,40.0,1
1587,Resort Hotel,26,0.0,0,26.0,1
1643,Resort Hotel,50,0.0,0,50.0,1
1752,Resort Hotel,26,0.0,0,26.0,1
1884,Resort Hotel,26,0.0,0,26.0,1
1917,Resort Hotel,27,0.0,0,27.0,1
1962,Resort Hotel,27,0.0,0,27.0,1
2003,Resort Hotel,26,0.0,0,26.0,1
2164,Resort Hotel,26,0.0,0,26.0,1
2173,Resort Hotel,55,0.0,0,55.0,1


In [54]:
#checing the customer types

df_clean[df_clean['adults'] > 10]['customer_type'].value_counts()


customer_type
Group    12
Name: count, dtype: int64

FINDINGS:
All 12 bookings with more than 10 adults belong to the Group customer category

#### CONCLUSION 

I came to the conclusion that, to keep all the 12 records because the cut type is group, which is a reasonabale explnation, a grp resrvtn may involve many guests

an outlier always cannot be an error

#### Checking the remaining missing values

we removed the 182 unusual rcords, but didnt filled the missing values

In [56]:
df_clean.isnull().sum()[df_clean.isnull().sum() > 0]

#filtering the new df count of null value

children             4
country            478
agent            16280
company         112440
total_guests         4
dtype: int64

#### Handle missing values in children

Earlier, I found that the most frequent value in children was 0. Since only four records are missing, i'll use 0 as planned replacement.

planned handling : fill with mode(0)

In [58]:
df_clean['children'] = df_clean['children'].fillna(0)

#fillna(0) replaces missing valueswith 0
#it updates the new children colm in df_clean


df_clean['children'].isnull().sum()
#checking how many mssing values remain

np.int64(0)

#### Calculating the total guests again

In [59]:
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']

df_clean['total_guests'].isnull().sum()

np.int64(0)

This happened by handling the missing values in children which had 4 missing values , now that we have filled it with 0 , total guest wouldnt ave any missing value

#### Handle missing values in country

Earlier, we found that the country column contains 478 missing values in our cleaned dataset

We could replace missing countries with PRT (Portugal), since it's the most frequent country.

But that would mean assuming those 478 bookings came from Portugal, even though we don't know their countries.

A better approach is to replace missing values with Unknown.

planned handling: Fill with Unknown

In [60]:
df_clean['country'] = df_clean['country'].fillna('Unknown')

df_clean['country'].isnull().sum()


np.int64(0)

#### Handle missing values in agents

- Missing agent IDs: 16,280
- Unique agent IDs: 333
- Agent IDs are identifiers, not numerical measurements.

decided to create a new column called has_agent rather than filling missing IDs with the mean or mode.

planned handling: Create has_agent

In [62]:
df_clean['has_agent'] = df_clean['agent'].notna().astype(int)
#notna() -> checks whether the agent id is availble true agent id is present else is missing
#.astype(int) -> true becomes 1, flse becomes 0 ,converts the boolean values into int
df_clean['has_agent'].value_counts()


has_agent
1    102928
0     16280
Name: count, dtype: int64

here, we are note deleting the agent col, we exclude from the model feature 

#### Handle missing values in children

Now applying  the same technique to the company column.

Earlier, i discovered:
- 112,440 missing company IDs
- 352 unique company IDs

Rather than replacing missing company IDs with an arbitrary number, I'll create has_company.

In [63]:
df_clean['has_company'] = df_clean['company'].notna().astype(int)

df_clean['has_company'].value_counts()


has_company
0    112440
1      6768
Name: count, dtype: int64

#### checking the remaining values

In [64]:
df_clean.isnull().sum()[df_clean.isnull().sum() > 0]


agent       16280
company    112440
dtype: int64

These missing values are expected because i created has_agent and has_company instead of filling the original ID columns.

I'll exclude the original agent and company columns when selecting features for ML model.

####  Removing the original ID columns ie agents and company

Since I've already created has_agent and has_company, I can remove the original columns from cleaned dataset.

Agent and company IDs are identifiers, not quantities. For example, agent 240 isn't necessarily more important than agent 9.

For first ML model, I'll use the simpler indicator features instead.

In [65]:
df_clean = df_clean.drop(columns=['agent', 'company'])

df_clean.shape


(119208, 33)

#### Final data quality check

In [66]:

print("Dataset Shape:", df_clean.shape)
print("Missing Values:", df_clean.isnull().sum().sum())
print("Duplicate Rows:", df_clean.duplicated().sum())
print("Zero Guest Bookings:", (df_clean['total_guests'] == 0).sum())
print("Minimum ADR:", df_clean['adr'].min())
print("Maximum ADR:", df_clean['adr'].max())


Dataset Shape: (119208, 33)
Missing Values: 0
Duplicate Rows: 31997
Zero Guest Bookings: 0
Minimum ADR: 0.0
Maximum ADR: 510.0


#### Observations

prevously they were 31,997 now its 31,994

We created new features and removed the original agent and company columns. Different records may now appear identical because their original IDs are no longer included.

This is not necessarily an error. We'll account for duplicate records when designing our train/test evaluation.

Why is the maximum ADR 510 instead of 1000?

Removed the extreme ADR value of 5400 using our chosen threshold of 1000. The highest remaining actual value happens to be 510.


## Data Cleaning Summary

- Original dataset: 119,390 rows and 32 columns.
- Removed 180 bookings with zero guests.
- Removed 2 bookings with unusual ADR values (-6.38 and 5400).
- Filled 4 missing `children` values with 0.
- Replaced missing `country` values with "Unknown".
- Created `total_guests` from adults, children, and babies.
- Created `has_agent` and `has_company` indicators.
- Removed the original `agent` and `company` ID columns.
- Retained 12 large group bookings with more than 10 adults.
- Retained duplicate rows because there is no unique booking identifier. Duplicate-related evaluation risks will be addressed during modeling.

### Final Dataset

- Rows: 119,208
- Columns: 33
- Missing values: 0
- ADR range: 0 to 510
- Zero-guest bookings: 0


In [68]:
df_clean.to_csv('../data/hotel_bookings_cleaned.csv', index=False)

#saving it as csv fle so can use it for ml wthout repeating all the cleaning steps

In [69]:

df_check = pd.read_csv('../data/hotel_bookings_cleaned.csv')

df_check.shape


(119208, 33)